In [0]:
# ===============================
# Setup Python Path for Src Folder
# ===============================
import sys
import os

# Calculate absolute path to Src folder relative to this notebook
src_path = os.path.abspath(os.path.join(os.getcwd(), "../Src"))
if src_path not in sys.path:
    sys.path.append(src_path)

print("Src folder added to Python path:", src_path)

# ===============================
# Import functions from Src
# ===============================
try:
    from DataContractValidator import load_contract, validate_table_contract
    from SilverLayer import create_silver_tables
except ModuleNotFoundError as e:
    print("Error importing modules from Src folder:", e)
    raise

# ===============================
# Load Contract YAML
# ===============================
contract_path = os.path.join(src_path, "Contracts.yaml")
contract = load_contract(contract_path)
print("Contract loaded successfully")

In [0]:
# =============================== 
# Imports & Setup 
# ===============================
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from collections import Counter

# Create schemas if they don't exist
spark.sql("DROP SCHEMA IF EXISTS bronze CASCADE")
spark.sql("DROP SCHEMA IF EXISTS silver CASCADE")
spark.sql("DROP SCHEMA IF EXISTS gold CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("Schemas created (bronze, silver, gold)")

In [0]:
# =============================== 
# Helper Functions 
# ===============================
def normalize_columns(df):
    """
    Normalize all column names to lowercase so Spark/Delta does not
    treat Quarter vs quarter as conflicting columns.
    """
    return df.toDF(*[c.lower() for c in df.columns])

def print_duplicate_columns(df, df_name):
    duplicates = [c for c, n in Counter(df.columns).items() if n > 1]
    print(f"{df_name} duplicate columns: {duplicates if duplicates else 'None'}")

In [0]:
# =============================== 
# Bronze Layer - Load FAERS Data 
# ===============================

# 2023
Demographics2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_demographics_2023"
)
Drug2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_drug_2023"
)
Reaction2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_drug_reaction_2023"
)
Outcome2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_patient_outcome_2023"
)

# 2022
Demographics2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_demographics_2022"
)
Drug2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_drug_2022"
)
Reaction2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_drug_reaction_2022"
)
Outcome2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_patient_outcome_2022"
)

# 2021
Demographics2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_demographics_2021"
)
Drug2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_drug_2021"
)
Reaction2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_drug_reaction_2021"
)
Outcome2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_patient_outcome_2021"
)

# Normalize Column Names
Demographics2021 = normalize_columns(Demographics2021)
Demographics2022 = normalize_columns(Demographics2022)
Demographics2023 = normalize_columns(Demographics2023)
Drug2021 = normalize_columns(Drug2021)
Drug2022 = normalize_columns(Drug2022)
Drug2023 = normalize_columns(Drug2023)
Reaction2021 = normalize_columns(Reaction2021)
Reaction2022 = normalize_columns(Reaction2022)
Reaction2023 = normalize_columns(Reaction2023)
Outcome2021 = normalize_columns(Outcome2021)
Outcome2022 = normalize_columns(Outcome2022)
Outcome2023 = normalize_columns(Outcome2023)
print("Column names normalized to lowercase")

# Fix data type
Drug2021_fixed = (
    Drug2021
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)
Drug2022_fixed = (
    Drug2022
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)
Drug2023_fixed = (
    Drug2023
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)

# Combine 3 years into single tables
Demographics = Demographics2021.unionByName(Demographics2022).unionByName(Demographics2023)
Drug = (Drug2021_fixed.unionByName(Drug2022_fixed, allowMissingColumns=True)
        .unionByName(Drug2023_fixed, allowMissingColumns=True))
Reaction = Reaction2021.unionByName(Reaction2022).unionByName(Reaction2023)
Outcome = Outcome2021.unionByName(Outcome2022).unionByName(Outcome2023)

# Keep only the latest version of each record in Demographics
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, col

windowSpec = Window.partitionBy("primary_id").orderBy(desc("case_version_number"))
Demographics = (Demographics
                .withColumn("row_num", row_number().over(windowSpec))
                .filter(col("row_num") == 1)
                .drop("row_num"))

# Save combined tables to bronze
Demographics.write.format("delta").mode("overwrite").saveAsTable("bronze.Demographics")
Drug.write.format("delta").mode("overwrite").saveAsTable("bronze.Drug")
Reaction.write.format("delta").mode("overwrite").saveAsTable("bronze.Reaction")
Outcome.write.format("delta").mode("overwrite").saveAsTable("bronze.Outcome")
print("Combined bronze tables created")

In [0]:
# =============================== 
# Run Data Contracts 
# ===============================
validate_table_contract(Demographics, "demographics", contract)
validate_table_contract(Drug, "drug", contract)
validate_table_contract(Reaction, "reaction", contract)
validate_table_contract(Outcome, "outcome", contract)
print("All data contracts passed")

In [0]:
# =============================== 
# Silver Layer 
# ===============================
silver_tables = create_silver_tables(Demographics, Drug, Reaction, Outcome)

# Demographics
silver_tables["demographics"].write.format("delta").mode("overwrite").saveAsTable("silver.demographics")
# Drugs
silver_tables["drugs"].write.format("delta").mode("overwrite").saveAsTable("silver.drugs")
# Reactions
silver_tables["reactions"].write.format("delta").mode("overwrite").saveAsTable("silver.reactions")
# Outcomes
silver_tables["outcomes"].write.format("delta").mode("overwrite").saveAsTable("silver.outcomes")
print("Silver tables created successfully")

In [0]:
# =============================== 
# Gold Layer 
# ===============================

# Load Silver tables from catalog
silver_demographics = spark.table("silver.demographics")
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")
silver_outcomes = spark.table("silver.outcomes")

In [0]:

#Top 5 Drugs with the Highest Adverse Event Reports Over Time

from pyspark.sql.functions import countDistinct, sum as spark_sum, col
from pyspark.sql.window import Window

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")

# Step 1: Compute yearly adverse event counts per drug
drug_event_counts = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .groupBy("year", "drug_name")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

# Step 2: Compute total reports per drug across all years
total_reports_per_drug = (
    drug_event_counts
    .groupBy("drug_name")
    .agg(spark_sum("num_reports").alias("total_reports"))
)

# Step 3: Get top 5 drugs by total reports
top5_drugs = (
    total_reports_per_drug
    .orderBy(col("total_reports").desc())
    .limit(5)
    .select("drug_name")
)

# Step 4: Filter yearly table to include only top 5 drugs
drug_event_counts_top5 = (
    drug_event_counts
    .join(top5_drugs, on="drug_name", how="inner")
    .orderBy("drug_name", "year")
)

# Step 5: Save as gold table
drug_event_counts_top5.write.format("delta").mode("overwrite").saveAsTable("gold.drug_event_counts_top5")

# Optional: display
display(drug_event_counts_top5)

In [0]:
# Most Common Drug Reactions

from pyspark.sql.functions import countDistinct, col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Load required tables
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")

# Get the top 5 drugs from previous gold table
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + reactions and keep only top 5 drugs
drug_reactions = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count reports per drug + reaction per year
reaction_counts = (
    drug_reactions
    .groupBy("year", "drug_name", "preferred_term_for_event")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

# Rank reactions per drug per year
windowSpec = Window.partitionBy("year", "drug_name").orderBy(col("num_reports").desc())

gold_top5_drug_reaction_trends = (
    reaction_counts
    .withColumn("rank", row_number().over(windowSpec))
    .filter(col("rank") <= 10)
)

# Save as gold table
gold_top5_drug_reaction_trends.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_reaction_trends")

display(gold_top5_drug_reaction_trends)

In [0]:
# Severe Outcomes by Drug

from pyspark.sql.functions import countDistinct, col

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_outcomes = spark.table("silver.outcomes")

# Load the Top 5 drugs from the previous gold table
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Severe outcome codes
severe_codes = ["DE", "HO", "LT"]

# Join drugs + outcomes and filter to Top 5 drugs
drug_outcomes = (
    silver_drugs
    .join(silver_outcomes, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count severe outcomes
gold_top5_drug_severe_outcomes = (
    drug_outcomes
    .filter(col("patient_outcome").isin(severe_codes))
    .groupBy("year", "drug_name", "patient_outcome")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year", "patient_outcome")
)

# Save gold table
gold_top5_drug_severe_outcomes.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_severe_outcomes")

display(gold_top5_drug_severe_outcomes)

In [0]:
# Adverse Events by Age Group
from pyspark.sql.functions import countDistinct

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_demographics = spark.table("silver.demographics")

# Load Top 5 drugs
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + demographics and filter to top 5 drugs
drug_demographics = (
    silver_drugs
    .join(silver_demographics, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Aggregate across all years
gold_top5_drug_age_groups = (
    drug_demographics
    .groupBy("drug_name", "age_group")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "num_reports")
)

# Save gold table
gold_top5_drug_age_groups.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_age_groups")

display(gold_top5_drug_age_groups)

In [0]:
# Adverse Events by Gender

from pyspark.sql.functions import countDistinct, col

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_demographics = spark.table("silver.demographics")

# Load Top 5 drugs
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + demographics and filter to Top 5 drugs
drug_gender = (
    silver_drugs
    .join(silver_demographics, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count reports by gender
gold_top5_drug_gender = (
    drug_gender
    .filter(col("gender_of_patient").isin("M", "F"))
    .groupBy("drug_name", "gender_of_patient")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "gender_of_patient")
)

# Save gold table
gold_top5_drug_gender.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_gender")

display(gold_top5_drug_gender)